# Week 2 · Class 3 — Building a LangGraph Agent

Build one real agent that reasons, does math, and searches the web — then extend it yourself.

## Before you start

This notebook assumes you already ran `week2_langchain_fundamentals.ipynb` in Class 2: you should
be comfortable with `ChatPromptTemplate`, `| llm` chains, and the `@tool` decorator. We do not
re-teach those here — we go straight to **LangGraph**, which Class 2 did not cover.

## How this notebook is organized

- **In class (Sections 1–14):** build one small, general-purpose agent — `chat`, `calculate`,
  `research`, and `docs`. Each intent is deliberately simple so the *routing pattern* is what
  sticks, not any one domain.
- **Homework challenge (Section 15):** design and add **one new intent of your own** — much less
  scaffolded than the in-class build. This is the harder part; you decide what it does.

Data collection for RAG is a **separate notebook**: `week2_data_collection_rag.ipynb` (Class 4).
That notebook wires a real chunk of scraped/PDF text into the `docs` node you build here.

Get a free API key at [console.groq.com](https://console.groq.com) before Section 2.

> Run cells **top to bottom**.


## Section 1 — Install packages


In [ ]:
!pip install -q langchain langchain-core langchain-groq langgraph langchain-community duckduckgo-search ddgs

## Section 2 — Groq API key

Use `getpass` in Colab — never commit API keys to Git.


In [ ]:
import os
import getpass

# Try Colab Secrets first; fall back to a prompt.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))


## Section 3 — Quick recap: what you already have

From Class 2 you already know how to build `ChatPromptTemplate` chains and a tool-calling agent.
The only new piece here is the LLM instance — everything else about prompts and tools carries
over unchanged.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq

MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL, temperature=0.7)

print("Model ready:", MODEL)


## Section 4 — Why LangGraph?

A **chain** (`prompt | llm`) always takes the same path. A **LangGraph agent** can take a
*different* path depending on what the user asks. Today's agent has four possible paths:

| Intent | What it does |
|--------|--------------|
| `chat` | General conversation, no tools |
| `calculate` | Runs a real arithmetic tool |
| `research` | Searches the web with a real tool |
| `docs` | Answers using reference text you provide (the seed of RAG) |


## Section 5 — Define state

A **StateGraph** holds shared data passed between nodes. Every node reads from and writes to this
one shared `AgentState`.


In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    intent: str
    context_chunks: str


## Section 6 — Classify and chat nodes (2 intents)

Start small: route between `chat` and nothing else yet. We'll add more intents one at a time so
you can see the pattern before it gets more complex.


In [ ]:
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the user message as exactly one word: chat or calculate.\n"
               "- chat: general conversation, questions, opinions\n"
               "- calculate: any arithmetic or math the user wants computed"),
    ("human", "{user_input}"),
])

def classify_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    user_text = last.content if hasattr(last, "content") else str(last)
    result = (classify_prompt | llm).invoke({"user_input": user_text})
    intent = result.content.strip().lower()
    return {"intent": "calculate" if "calculate" in intent else "chat"}

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly, helpful assistant. Answer clearly in under 5 sentences."),
    ("human", "{user_input}"),
])

def chat_node(state: AgentState) -> dict:
    result = (chat_prompt | llm).invoke({"user_input": state["messages"][-1].content})
    return {"messages": [AIMessage(content=result.content)]}


## Section 7 — The `calculate` node (your first tool)

A node doesn't have to just call `llm` — it can run a **tool** first, then hand the result to the
LLM to turn into a proper sentence.


In [ ]:
from langchain.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression. Only numbers and + - * / ( )."""
    allowed = set("0123456789+-*/()., ")
    if not set(expression).issubset(allowed):
        return "Error: expression contains unsafe characters"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

calc_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are given a math question and its computed answer. State the answer clearly in one sentence."),
    ("human", "Question: {user_input}\nComputed result: {result}"),
])

def calculate_node(state: AgentState) -> dict:
    user_text = state["messages"][-1].content
    # naive extraction: pull out digits/operators; good enough for this demo
    expr = "".join(ch for ch in user_text if ch in "0123456789+-*/(). ")
    computed = calculate.invoke(expr.strip()) if expr.strip() else "unable to parse expression"
    result = (calc_prompt | llm).invoke({"user_input": user_text, "result": computed})
    return {"messages": [AIMessage(content=result.content)]}


## Section 8 — Build and test the 2-intent graph


In [ ]:
def route(state: AgentState) -> str:
    return state["intent"]

graph = StateGraph(AgentState)
graph.add_node("classify", classify_node)
graph.add_node("chat", chat_node)
graph.add_node("calculate", calculate_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {"chat": "chat", "calculate": "calculate"})
graph.add_edge("chat", END)
graph.add_edge("calculate", END)

agent = graph.compile()

for prompt in ["What's a good icebreaker question?", "What is 42 * 17?"]:
    result = agent.invoke({"messages": [HumanMessage(content=prompt)], "intent": "", "context_chunks": ""})
    print(f"\n--- '{prompt}' → intent: {result['intent']}")
    print(result["messages"][-1].content[:200])


## Section 9 — Add a `research` node (a second kind of tool)

`calculate` runs a deterministic tool. `research` runs a **search** tool — same pattern, different
tool. This is the node that lets the agent know things past its training data.


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

research_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using the search result below. Mention it comes from a web search. "
               "If the result doesn't answer the question, say so honestly."),
    ("human", "Question: {user_input}\n\nSearch result:\n{search_result}"),
])

def research_node(state: AgentState) -> dict:
    user_text = state["messages"][-1].content
    search_result = search.run(user_text)
    result = (research_prompt | llm).invoke({"user_input": user_text, "search_result": search_result[:800]})
    return {"messages": [AIMessage(content=result.content)]}

# classify now has to know a third option exists
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the user message as exactly one word: chat, calculate, or research.\n"
               "- chat: general conversation, opinions, things the model likely already knows\n"
               "- calculate: any arithmetic or math the user wants computed\n"
               "- research: needs a current fact, statistic, or something to look up"),
    ("human", "{user_input}"),
])

def classify_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    user_text = last.content if hasattr(last, "content") else str(last)
    result = (classify_prompt | llm).invoke({"user_input": user_text})
    intent = result.content.strip().lower()
    if "calculate" in intent:
        intent = "calculate"
    elif "research" in intent:
        intent = "research"
    else:
        intent = "chat"
    return {"intent": intent}

graph = StateGraph(AgentState)
graph.add_node("classify", classify_node)
graph.add_node("chat", chat_node)
graph.add_node("calculate", calculate_node)
graph.add_node("research", research_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {"chat": "chat", "calculate": "calculate", "research": "research"})
graph.add_edge("chat", END)
graph.add_edge("calculate", END)
graph.add_edge("research", END)

agent = graph.compile()
print("Agent upgraded with a research node.")


## Section 10 — Test all three intents


In [ ]:
for prompt in [
    "What's a good icebreaker question?",
    "What is 42 * 17?",
    "Search for who founded Groq",
]:
    result = agent.invoke({"messages": [HumanMessage(content=prompt)], "intent": "", "context_chunks": ""})
    print(f"\n--- '{prompt}' → intent: {result['intent']}")
    print(result["messages"][-1].content[:200])


## Section 11 — Add a `docs` node (context-aware answers)

None of the nodes so far can read **your** documents. The `docs` node answers using whatever text
is sitting in `context_chunks` — right now that's empty, so we'll test it with a placeholder.
Class 4 wires real scraped/PDF text into this exact field.


In [ ]:
docs_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the reference text below. "
               "If the answer isn't in the text, say you don't have that information."),
    ("human", "Reference text:\n{context}\n\nQuestion: {user_input}"),
])

def docs_node(state: AgentState) -> dict:
    context = state.get("context_chunks") or "(no reference text provided)"
    result = (docs_prompt | llm).invoke({"user_input": state["messages"][-1].content, "context": context})
    return {"messages": [AIMessage(content=result.content)]}

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the user message as exactly one word: chat, calculate, research, or docs.\n"
               "- chat: general conversation, opinions\n"
               "- calculate: any arithmetic the user wants computed\n"
               "- research: needs a current fact to look up online\n"
               "- docs: asks about \"this document\", \"the text\", or provided reference material"),
    ("human", "{user_input}"),
])

def classify_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    user_text = last.content if hasattr(last, "content") else str(last)
    result = (classify_prompt | llm).invoke({"user_input": user_text})
    intent = result.content.strip().lower()
    if "calculate" in intent:
        intent = "calculate"
    elif "research" in intent:
        intent = "research"
    elif "docs" in intent:
        intent = "docs"
    else:
        intent = "chat"
    return {"intent": intent}

graph = StateGraph(AgentState)
graph.add_node("classify", classify_node)
graph.add_node("chat", chat_node)
graph.add_node("calculate", calculate_node)
graph.add_node("research", research_node)
graph.add_node("docs", docs_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {
    "chat": "chat", "calculate": "calculate", "research": "research", "docs": "docs",
})
graph.add_edge("chat", END)
graph.add_edge("calculate", END)
graph.add_edge("research", END)
graph.add_edge("docs", END)

agent = graph.compile()
print("Agent upgraded with a docs node.")


## Section 12 — Test the docs route


In [ ]:
placeholder_context = "Groq is a company that builds fast AI inference hardware called LPUs (Language Processing Units)."

result = agent.invoke({
    "messages": [HumanMessage(content="What does this document say Groq builds?")],
    "intent": "",
    "context_chunks": placeholder_context,
})
print("Intent:", result["intent"])
print(result["messages"][-1].content)

# Try a question the context doesn't answer — the agent should say it doesn't know
result2 = agent.invoke({
    "messages": [HumanMessage(content="According to this document, when was Groq founded?")],
    "intent": "",
    "context_chunks": placeholder_context,
})
print("\nIntent:", result2["intent"])
print(result2["messages"][-1].content)


## Section 13 — Single-agent checkpoint

You now have **one working agent** with four intents:
- `chat` — plain conversation
- `calculate` — a deterministic tool
- `research` — a live web-search tool
- `docs` — answers grounded in text you provide (currently a placeholder — Class 4 makes it real)

Test a few of your own prompts across all four before moving on.


In [ ]:
# Your turn: try your own prompt
custom = "What's 15% of 240?"
result = agent.invoke({"messages": [HumanMessage(content=custom)], "intent": "", "context_chunks": ""})
print("Intent:", result["intent"])
print(result["messages"][-1].content)


## Section 14 — Why does the agent need real data?

`docs` only works as well as whatever you put in `context_chunks`. Right now that's one made-up
sentence. **Class 4** (separate notebook: `week2_data_collection_rag.ipynb`) scrapes real web
pages and parses a real PDF, then feeds real chunks into this exact node.


## Section 15 — Homework challenge: design your own intent

**This is the harder part.** In class you followed a scaffold for four intents. For homework,
**design and add a fifth intent of your own** — with much less hand-holding.

**Requirements:**
1. Pick a capability that isn't already covered (not another calculator or search wrapper)
2. Update the classify prompt so it can correctly choose your new intent
3. Write the node function — it can call a tool, call `llm` directly, or both
4. Wire the new node + edge into the graph and test it with 2–3 prompts

**Ideas to get you started (pick one, or bring your own):**
- `translate` — detect a target language in the request and translate the text
- `summarize` — condense a long pasted paragraph into 2–3 sentences
- `recommend` — suggest something (a book, a tool, a next step) based on a short description
- `format` — rewrite the user's text into a specific structure (bullet list, table, email)

No scaffold is provided below — this is intentionally a blank cell. Use Sections 6–11 as your
reference for the *pattern* (prompt → node function → update classify → rebuild graph → reassign
`agent`), but the specifics are yours to design.


In [ ]:
# Your homework: build it here.
# Reminders:
#   - reuse `llm` — don't redefine it
#   - your node function signature must be: def my_node(state: AgentState) -> dict
#   - it must return {"messages": [AIMessage(content=...)]}
#   - after rebuilding the graph, reassign: agent = <your compiled graph>
